In [ ]:
# ==================== 配置开关 ====================
FORCE_RECOMPUTE_FEATURES = True
FORCE_RETRAIN = True
FORCE_RECREATE_FIGS = True

# ==================== 路径设置 ====================
import os
from pathlib import Path

USE_TEST = 0
VERSION_DIR = Path.cwd()

FIG_DIR = VERSION_DIR / "fig"
MODEL_DIR = VERSION_DIR / "model"
FEATURE_DIR = VERSION_DIR / "featured_data"

ROOT_DIR = VERSION_DIR.parent
DATA_DIR = ROOT_DIR / "data"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

for dir_path in [FIG_DIR, MODEL_DIR, FEATURE_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

print(f"当前版本文件夹: {VERSION_DIR}")
print(f"图片保存目录: {FIG_DIR}")
print(f"模型保存目录: {MODEL_DIR}")
print(f"特征数据保存目录: {FEATURE_DIR}")
print(f"训练数据路径: {TRAIN_PATH}")
print(f"测试数据路径: {TEST_PATH}")
print("提交样例文件仅作格式参考，不参与运行")

# ==================== Layer3 WSL temp strategy ====================
# This config is self-contained and does not depend on Layer2 helpers.
# Usage:
# 1. Default is teammate's WSL external drive: "e"
# 2. Change to "g" on your WSL machine if your external drive is mounted at ./scratch/g
# 3. Only use drive letters here, not "E:" and not "./scratch/e"

L3_TEMP_DRIVE_LETTER = "e"
L3_TEMP_BACKUP_DRIVE_LETTERS = ["g"]
L3_TEMP_DIR_NAME = "layer3_temp"
L3_TEMP_FALLBACK_ROOT = VERSION_DIR / "layer3_temp"


def normalize_drive_letter(letter):
    return str(letter).strip().lower().replace(":", "").replace("\\", "").replace("/", "")


def wsl_drive_path(letter, dir_name):
    letter = normalize_drive_letter(letter)
    if not letter:
        raise ValueError("drive letter cannot be empty")
    return Path(f"./scratch/{letter}/{dir_name}")


def build_drive_candidates(primary_letter, backup_letters, dir_name, fallback_root):
    ordered_letters = []
    for letter in [primary_letter, *(backup_letters or [])]:
        norm = normalize_drive_letter(letter)
        if norm and norm not in ordered_letters:
            ordered_letters.append(norm)

    candidates = [wsl_drive_path(letter, dir_name) for letter in ordered_letters]
    candidates.append(Path(fallback_root))
    return candidates


def choose_writable_root(candidates):
    last_error = None

    for path in candidates:
        try:
            path.mkdir(parents=True, exist_ok=True)
            probe = path / ".write_test"
            probe.write_text("ok", encoding="utf-8")
            probe.unlink()
            return path
        except Exception as e:
            last_error = e
            print(f"[layer3 temp] unavailable: {path} -> {e}")

    raise RuntimeError(f"no writable Layer3 temp directory found, last error: {last_error}")


L3_TEMP_CANDIDATES = build_drive_candidates(
    primary_letter=L3_TEMP_DRIVE_LETTER,
    backup_letters=L3_TEMP_BACKUP_DRIVE_LETTERS,
    dir_name=L3_TEMP_DIR_NAME,
    fallback_root=L3_TEMP_FALLBACK_ROOT,
)

L3_TEMP_ROOT = choose_writable_root(L3_TEMP_CANDIDATES)

print("\n=== Layer3 temp strategy ===")
print(f"L3_TEMP_DRIVE_LETTER: {L3_TEMP_DRIVE_LETTER}")
print(f"L3_TEMP_BACKUP_DRIVE_LETTERS: {L3_TEMP_BACKUP_DRIVE_LETTERS}")
print(f"L3_TEMP_DIR_NAME: {L3_TEMP_DIR_NAME}")
print("L3 temp candidates:")
for p in L3_TEMP_CANDIDATES:
    print(f" - {p}")
print(f"L3_TEMP_ROOT: {L3_TEMP_ROOT}")

# ==================== Text Version Switch ====================
# v1: 原始均值向量
# v2: max pooling
# v3: tf-idf weighted
TEXT_VERSION = "v1"   # 可选: "v1" / "v2" / "v3"

TEXT_VERSION_CONFIGS = {
    "v1": {
        "label": "V1",
        "train_path": FEATURE_DIR / "final_train.parquet",
        "test_path": FEATURE_DIR / "final_test.parquet",
        "title_col": "title_w2v_features",
        "comment_col": "comment_w2v_features",
    },
    "v2": {
        "label": "V2",
        "train_path": VERSION_DIR / "featured_data_max_pooling" / "final_improved_train.parquet",
        "test_path": VERSION_DIR / "featured_data_max_pooling" / "final_improved_test.parquet",
        "title_col": "title_w2v_max",
        "comment_col": "comment_w2v_max",
    },
    "v3": {
        "label": "V3",
        "train_path": FEATURE_DIR / "final_improved_train.parquet",
        "test_path": FEATURE_DIR / "final_improved_test.parquet",
        "title_col": "title_w2v_tfidf",
        "comment_col": "comment_w2v_tfidf",
    },
}

if TEXT_VERSION not in TEXT_VERSION_CONFIGS:
    raise ValueError(f"unknown TEXT_VERSION: {TEXT_VERSION}")

TEXT_VERSION_CONFIG = TEXT_VERSION_CONFIGS[TEXT_VERSION]

print("\n=== Text Version Switch ===")
print(f"TEXT_VERSION: {TEXT_VERSION}")
print(f"TEXT_LABEL: {TEXT_VERSION_CONFIG['label']}")
print(f"L2 train path: {TEXT_VERSION_CONFIG['train_path']}")
print(f"L2 test path: {TEXT_VERSION_CONFIG['test_path']}")
print(f"title vector col: {TEXT_VERSION_CONFIG['title_col']}")
print(f"comment vector col: {TEXT_VERSION_CONFIG['comment_col']}")

# ==================== Model Switch ====================
# x: XGBoost
# l: LightGBM
MODEL_TYPE = "l"   # 可选: "x" / "l"

if MODEL_TYPE not in {"x", "l"}:
    raise ValueError(f"unknown MODEL_TYPE: {MODEL_TYPE}")

MODEL_TAG = "xgb" if MODEL_TYPE == "x" else "lgb"
MODEL_FILE_EXT = "json" if MODEL_TYPE == "x" else "txt"

print("\n=== Model Switch ===")
print(f"MODEL_TYPE: {MODEL_TYPE}")
print(f"MODEL_TAG: {MODEL_TAG}")



In [ ]:
# spark.stop()

In [ ]:
# ==================== Spark 初始化 ====================
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import broadcast

# 直接在 config 中引用代码块一的 L3_TEMP_ROOT
spark = SparkSession.builder \
    .appName("Layer3_Features_Production") \
    .master("local[16]") \
    .config("spark.driver.memory", "14g") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "4g") \
    .config("spark.sql.shuffle.partitions", "400") \
    .config("spark.default.parallelism", "400") \
    .config("spark.sql.autoBroadcastJoinThreshold", "50mb") \
    .config("spark.memory.fraction", "0.6") \
    .config("spark.memory.storageFraction", "0.3") \
    .config("spark.network.timeout", "800s") \
    .config("spark.executor.heartbeatInterval", "60s") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.rpc.message.maxSize", "1024") \
    .config("spark.local.dir", str(L3_TEMP_ROOT)) \
    .getOrCreate()

print("✅ Spark 稳定性配置已成功加载：")
print(f"- 核心分配: 使用 16 个线程 (local[16])")
print(f"- 内存配置: 14G (Heap) + 4G (Off-Heap)")
print(f"- 运行临时目录: {spark.conf.get('spark.local.dir')}")

In [ ]:
from pyspark.sql import functions as F


class GoodsFeatures:
    """
    Layer3 feature engineering:
    1. explicit input column selection
    2. early numeric cleanup
    3. time features + user/product profiles + LOO
    4. strict validation mode avoids leakage by splitting before profile stats
    """

    TRAIN_REQUIRED_COLS = [
        "id", "user_id", "prod_id", "time", "votes", "purchased", "rating",
        "title_w2v_tfidf", "comment_w2v_tfidf",
    ]
    TEST_REQUIRED_COLS = [
        "id", "user_id", "prod_id", "time", "votes", "purchased",
        "title_w2v_tfidf", "comment_w2v_tfidf",
    ]

    def __init__(
        self,
        spark,
        train_input_path,
        test_input_path,
        train_output_path,
        test_output_path,
        smoothing_strength=10,
        log_base=2.718,
        force_recompute=False,
    ):
        self.spark = spark
        self.train_input_path = Path(train_input_path)
        self.test_input_path = Path(test_input_path)
        self.train_output_path = Path(train_output_path)
        self.test_output_path = Path(test_output_path)
        self.alpha = smoothing_strength
        self.e = log_base
        self.force_recompute = force_recompute

        self.train_input_count = None
        self.test_input_count = None

    def _read_selected(self, path, required_cols):
        df = self.spark.read.parquet(str(path))
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            raise ValueError(f"{path} missing required cols: {missing}")
        return df.select(*required_cols)

    def _clean_base_df(self, df, is_train):
        tag = "train" if is_train else "test"

        before_count = df.count()
        print(f"[{tag}] raw input rows: {before_count}")

        df = df.filter(F.col("user_id").isNotNull() & F.col("prod_id").isNotNull())
        after_id_filter = df.count()
        print(f"[{tag}] after user_id/prod_id filter: {after_id_filter} (removed {before_count - after_id_filter})")

        df = df.withColumn("votes", F.coalesce(F.col("votes").cast("double"), F.lit(0.0)))
        df = df.withColumn(
            "purchased",
            F.when(
                F.lower(F.col("purchased").cast("string")).isin("true", "1"),
                F.lit(1.0),
            ).otherwise(F.lit(0.0))
        )
        df = df.withColumn("time_ts", F.to_timestamp("time"))

        bad_time_count = df.filter(F.col("time").isNotNull() & F.col("time_ts").isNull()).count()
        print(f"[{tag}] bad time rows: {bad_time_count}")

        if is_train:
            df = df.withColumn("rating", F.col("rating").cast("double"))
            before_rating_filter = df.count()
            df = df.filter(F.col("rating").isin([1.0, 2.0, 3.0, 4.0, 5.0]))
            after_rating_filter = df.count()
            print(f"[{tag}] after rating filter: {after_rating_filter} (removed {before_rating_filter - after_rating_filter})")

        return df

    def load_data(self):
        train_df = self._read_selected(self.train_input_path, self.TRAIN_REQUIRED_COLS)
        test_df = self._read_selected(self.test_input_path, self.TEST_REQUIRED_COLS)

        train_df = self._clean_base_df(train_df, is_train=True)
        test_df = self._clean_base_df(test_df, is_train=False)

        self.train_input_count = train_df.count()
        self.test_input_count = test_df.count()

        print(f"[train] cleaned input rows: {self.train_input_count}")
        print(f"[test] cleaned input rows: {self.test_input_count}")

        return train_df, test_df

    def load_train_only_data(self):
        train_df = self._read_selected(self.train_input_path, self.TRAIN_REQUIRED_COLS)
        train_df = self._clean_base_df(train_df, is_train=True)

        self.train_input_count = train_df.count()
        print(f"[train] cleaned input rows: {self.train_input_count}")

        return train_df

    def compute_global_stats(self, train_df):
        global_avg = float(train_df.select(F.avg("rating")).collect()[0][0])

        user_counts = train_df.groupBy("user_id").count()
        tau = user_counts.approxQuantile("count", [0.5], 0.05)[0]
        if tau is None or tau <= 0:
            tau = 1.0

        print(f"[stats] global_avg={global_avg:.4f}, tau={tau:.4f}")
        return {
            "global_avg": global_avg,
            "tau": tau,
        }

    def build_user_features(self, train_df, global_stats):
        mu = global_stats["global_avg"]

        user_feat = (
            train_df.groupBy("user_id")
            .agg(
                F.count("rating").alias("u_cnt"),
                F.sum("rating").alias("u_sum_rating"),
                F.avg("votes").alias("u_avg_votes_received"),
            )
            .withColumn(
                "u_avg_smoothed_full",
                (F.col("u_sum_rating") + self.alpha * mu) / (F.col("u_cnt") + self.alpha),
            )
        )
        return user_feat

    def build_product_features(self, train_df, global_stats):
        mu = global_stats["global_avg"]

        df_weighted = train_df.withColumn("w", F.log(F.lit(self.e) + F.col("votes")))

        prod_feat = (
            df_weighted.groupBy("prod_id")
            .agg(
                F.count("rating").alias("p_cnt"),
                F.sum(F.col("rating") * F.col("w")).alias("p_sum_w"),
                F.sum("w").alias("p_sum_weight"),
            )
            .withColumn(
                "p_avg_smoothed_full",
                (F.col("p_sum_w") + self.alpha * mu) / (F.col("p_sum_weight") + self.alpha),
            )
        )
        return prod_feat

    def add_time(self, df, is_test=False, item_first_time=None):
        df = (
            df.withColumn("review_month", F.coalesce(F.month("time_ts"), F.lit(0)))
            .withColumn("day_of_week", F.coalesce(F.dayofweek("time_ts"), F.lit(0)))
        )

        if not is_test:
            item_first_time = df.groupBy("prod_id").agg(F.min("time_ts").alias("p_first_time"))

        df = df.join(item_first_time, "prod_id", "left")

        raw_age_days = F.when(
            F.col("time_ts").isNull() | F.col("p_first_time").isNull(),
            F.lit(0.0),
        ).otherwise(
            (F.col("time_ts").cast("long") - F.col("p_first_time").cast("long")) / 86400.0
        )

        df = df.withColumn(
            "prod_age_days",
            F.greatest(raw_age_days.cast("double"), F.lit(0.0)),
        )

        return df.drop("p_first_time"), item_first_time

    def LOO(self, train_df, global_stats):
        mu = global_stats["global_avg"]
        train_df = train_df.withColumn("w", F.log(F.lit(self.e) + F.col("votes")))

        train_df = train_df.withColumn(
            "u_avg_smoothed",
            ((F.col("u_sum_rating") - F.col("rating")) + self.alpha * mu) /
            ((F.col("u_cnt") - F.lit(1.0)) + self.alpha),
        )

        train_df = train_df.withColumn(
            "p_avg_smoothed",
            ((F.col("p_sum_w") - F.col("rating") * F.col("w")) + self.alpha * mu) /
            ((F.col("p_sum_weight") - F.col("w")) + self.alpha),
        )

        return train_df.drop(
            "u_sum_rating",
            "p_sum_w",
            "p_sum_weight",
            "w",
            "u_avg_smoothed_full",
            "p_avg_smoothed_full",
        )

    def build_user_product_features(self, df, global_stats):
        tau = global_stats["tau"]

        votes_safe = F.greatest(F.coalesce(F.col("votes").cast("double"), F.lit(0.0)), F.lit(0.0))
        u_cnt_safe = F.greatest(F.coalesce(F.col("u_cnt").cast("double"), F.lit(0.0)), F.lit(0.0))
        p_cnt_safe = F.greatest(F.coalesce(F.col("p_cnt").cast("double"), F.lit(0.0)), F.lit(0.0))

        df = df.withColumn("expectation_gap", F.col("p_avg_smoothed") - F.col("u_avg_smoothed"))
        df = df.withColumn("abs_expectation_gap", F.abs(F.col("expectation_gap")))
        df = df.withColumn("u_trust", 1 - F.exp(-u_cnt_safe / F.lit(float(tau))))
        df = df.withColumn("is_super_user", (u_cnt_safe > F.lit(float(tau) * 5.0)).cast("int"))

        df = df.withColumn("log_votes", F.log(F.lit(1.0) + votes_safe))
        df = df.withColumn("log_u_cnt", F.log(F.lit(1.0) + u_cnt_safe))
        df = df.withColumn("log_p_cnt", F.log(F.lit(1.0) + p_cnt_safe))

        return df

    def _validate_output_df(self, df, name, expected_count, is_train):
        total_count = df.count()
        unique_count = df.select("id").distinct().count()

        print(f"[{name}] total={total_count}, unique_id={unique_count}")

        if total_count != unique_count:
            raise ValueError(f"{name} contains duplicate ids")
        if expected_count is not None and total_count != expected_count:
            raise ValueError(f"{name} row count mismatch: expected {expected_count}, got {total_count}")

        if is_train:
            valid_rating_count = df.filter(F.col("rating").isin([1.0, 2.0, 3.0, 4.0, 5.0])).count()
            if valid_rating_count != total_count:
                raise ValueError(f"{name} contains invalid rating values")

    def _finalize_output_df(self, df, global_stats):
        return (
            df.drop("time_ts")
            .fillna({
                "purchased": 0.0,
                "review_month": 0,
                "day_of_week": 0,
                "prod_age_days": 0.0,
                "u_cnt": 0,
                "u_trust": 0.0,
                "u_avg_smoothed": global_stats["global_avg"],
                "u_avg_votes_received": 0.0,
                "is_super_user": 0,
                "p_cnt": 0,
                "p_avg_smoothed": global_stats["global_avg"],
                "expectation_gap": 0.0,
                "abs_expectation_gap": 0.0,
                "log_votes": 0.0,
                "log_u_cnt": 0.0,
                "log_p_cnt": 0.0,
            })
        )

    def _split_train_by_time(self, train_df, holdout_ratio=0.1):
        if not (0.0 < holdout_ratio < 1.0):
            raise ValueError(f"holdout_ratio must be in (0, 1), got {holdout_ratio}")

        split_quantile = 1.0 - float(holdout_ratio)

        train_df = train_df.withColumn(
            "time_key",
            F.coalesce(F.col("time_ts").cast("long"), F.lit(0)),
        )

        split_ts = train_df.approxQuantile("time_key", [split_quantile], 0.001)[0]
        if split_ts is None:
            raise ValueError("failed to compute split timestamp for strict validation")

        train_core_raw = train_df.filter(F.col("time_key") <= F.lit(split_ts)).drop("time_key")
        val_holdout_raw = train_df.filter(F.col("time_key") > F.lit(split_ts)).drop("time_key")

        train_core_count = train_core_raw.count()
        val_count = val_holdout_raw.count()

        print(f"[strict split] split_ts={split_ts}")
        print(f"[strict split] train_core={train_core_count}, val_holdout={val_count}")

        if train_core_count == 0:
            raise ValueError("train_core is empty after strict split")
        if val_count == 0:
            raise ValueError("val_holdout is empty after strict split")
        if self.train_input_count is not None and (train_core_count + val_count != self.train_input_count):
            raise ValueError(
                f"strict split row mismatch: train_core + val_holdout = {train_core_count + val_count}, "
                f"expected cleaned train count = {self.train_input_count}"
            )

        return train_core_raw, val_holdout_raw, train_core_count, val_count

    def run(self):
        print(">>> 1. load data...")
        train_df, test_df = self.load_data()

        print(">>> 2. add time features...")
        train_df, item_first_time = self.add_time(train_df, is_test=False)
        test_df, _ = self.add_time(test_df, is_test=True, item_first_time=item_first_time)

        print(">>> 3. compute global stats and profile tables...")
        stats = self.compute_global_stats(train_df)
        user_feat = self.build_user_features(train_df, stats)
        prod_feat = self.build_product_features(train_df, stats)

        print(">>> 4. build train with LOO...")
        train_df = train_df.join(user_feat, "user_id", "left").join(prod_feat, "prod_id", "left")
        train_df = self.LOO(train_df, stats)
        train_df = self.build_user_product_features(train_df, stats)

        print(">>> 5. build test by mapping full profiles...")
        user_feat_test = user_feat.drop("u_sum_rating").withColumnRenamed("u_avg_smoothed_full", "u_avg_smoothed")
        prod_feat_test = prod_feat.drop("p_sum_w", "p_sum_weight").withColumnRenamed("p_avg_smoothed_full", "p_avg_smoothed")

        test_df = test_df.join(user_feat_test, "user_id", "left").join(prod_feat_test, "prod_id", "left")
        test_df = test_df.fillna({
            "u_avg_smoothed": stats["global_avg"],
            "p_avg_smoothed": stats["global_avg"],
            "u_cnt": 0,
            "p_cnt": 0,
            "u_avg_votes_received": 0.0,
        })
        test_df = self.build_user_product_features(test_df, stats)

        print(">>> 6. finalize outputs...")
        train_df = self._finalize_output_df(train_df, stats)
        test_df = self._finalize_output_df(test_df, stats)

        print(">>> 7. validate outputs...")
        self._validate_output_df(train_df, "layer3-train", self.train_input_count, is_train=True)
        self._validate_output_df(test_df, "layer3-test", self.test_input_count, is_train=False)

        print(">>> 8. save outputs...")
        self.train_output_path.parent.mkdir(parents=True, exist_ok=True)
        self.test_output_path.parent.mkdir(parents=True, exist_ok=True)
        train_df.write.mode("overwrite").parquet(str(self.train_output_path))
        test_df.write.mode("overwrite").parquet(str(self.test_output_path))
        print("✅ Layer 3 feature build finished")

    def run_strict_validation(self, train_core_output_path, val_output_path, holdout_ratio=0.1):
        print(">>> 1. load train data...")
        train_df = self.load_train_only_data()

        print(">>> 2. strict split by time...")
        train_core_raw, val_holdout_raw, train_core_count, val_count = self._split_train_by_time(
            train_df,
            holdout_ratio=holdout_ratio,
        )

        print(">>> 3. add time features...")
        train_core_df, item_first_time = self.add_time(train_core_raw, is_test=False)
        val_holdout_df, _ = self.add_time(val_holdout_raw, is_test=True, item_first_time=item_first_time)

        print(">>> 4. compute train_core stats and profiles...")
        stats = self.compute_global_stats(train_core_df)
        user_feat = self.build_user_features(train_core_df, stats)
        prod_feat = self.build_product_features(train_core_df, stats)

        print(">>> 5. build train_core with LOO...")
        train_core_df = train_core_df.join(user_feat, "user_id", "left").join(prod_feat, "prod_id", "left")
        train_core_df = self.LOO(train_core_df, stats)
        train_core_df = self.build_user_product_features(train_core_df, stats)

        print(">>> 6. build val_holdout by mapping train_core full profiles...")
        user_feat_val = user_feat.drop("u_sum_rating").withColumnRenamed("u_avg_smoothed_full", "u_avg_smoothed")
        prod_feat_val = prod_feat.drop("p_sum_w", "p_sum_weight").withColumnRenamed("p_avg_smoothed_full", "p_avg_smoothed")

        val_holdout_df = val_holdout_df.join(user_feat_val, "user_id", "left").join(prod_feat_val, "prod_id", "left")
        val_holdout_df = val_holdout_df.fillna({
            "u_avg_smoothed": stats["global_avg"],
            "p_avg_smoothed": stats["global_avg"],
            "u_cnt": 0,
            "p_cnt": 0,
            "u_avg_votes_received": 0.0,
        })
        val_holdout_df = self.build_user_product_features(val_holdout_df, stats)

        print(">>> 7. finalize outputs...")
        train_core_df = self._finalize_output_df(train_core_df, stats)
        val_holdout_df = self._finalize_output_df(val_holdout_df, stats)

        print(">>> 8. validate outputs...")
        self._validate_output_df(train_core_df, "layer3-train-core", train_core_count, is_train=True)
        self._validate_output_df(val_holdout_df, "layer3-val-holdout", val_count, is_train=True)

        train_core_path = Path(train_core_output_path)
        val_path = Path(val_output_path)
        train_core_path.parent.mkdir(parents=True, exist_ok=True)
        val_path.parent.mkdir(parents=True, exist_ok=True)

        print(">>> 9. save strict validation outputs...")
        train_core_df.write.mode("overwrite").parquet(str(train_core_path))
        val_holdout_df.write.mode("overwrite").parquet(str(val_path))
        print("✅ Strict validation feature build finished")


In [ ]:
# ==================== Bridge cell: run Layer 3 feature engineering ====================
import time

# --- 1. 开始预处理计时 ---
prep_start = time.time()

TITLE_VECTOR_COL = TEXT_VERSION_CONFIG["title_col"]
COMMENT_VECTOR_COL = TEXT_VERSION_CONFIG["comment_col"]

L2_TRAIN_PATH = TEXT_VERSION_CONFIG["train_path"]
L2_TEST_PATH = TEXT_VERSION_CONFIG["test_path"]

L3_TRAIN_OUT = FEATURE_DIR / f"layer3_train_{TEXT_VERSION}.parquet"
L3_TEST_OUT = FEATURE_DIR / f"layer3_test_{TEXT_VERSION}.parquet"
L3_TRAIN_CORE_OUT = FEATURE_DIR / f"layer3_train_core_{TEXT_VERSION}.parquet"
L3_VAL_HOLDOUT_OUT = FEATURE_DIR / f"layer3_val_holdout_{TEXT_VERSION}.parquet"

STRICT_HOLDOUT_RATIO = 0.1

L2_TRAIN_REQUIRED_COLS = [
    "id", "user_id", "prod_id", "time", "votes", "purchased", "rating",
    "title_tfidf_features", "comment_tfidf_features",
    TITLE_VECTOR_COL, COMMENT_VECTOR_COL,
]
L2_TEST_REQUIRED_COLS = [
    "id", "user_id", "prod_id", "time", "votes", "purchased",
    "title_tfidf_features", "comment_tfidf_features",
    TITLE_VECTOR_COL, COMMENT_VECTOR_COL,
]

# --- 2. 结束预处理计时并记录变量 ---
prep_end = time.time()
total_prep_time_sec = prep_end - prep_start
print(f"\n[Efficiency] 特征工程/预处理完成，总耗时: {total_prep_time_sec:.2f} 秒")

def validate_l2_input(path, required_cols, is_train=False, expected_count=None):
    if not path.exists():
        raise FileNotFoundError(f"Layer2 input file not found: {path}")

    df = spark.read.parquet(str(path))

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{path} missing required cols: {missing}")

    df = df.select(*required_cols)

    total_count = df.count()
    unique_count = df.select("id").distinct().count()
    print(f"[L2 {'train' if is_train else 'test'}] total={total_count}, unique_id={unique_count}")

    if total_count != unique_count:
        raise ValueError(f"{path} contains duplicate ids")

    if expected_count is not None and total_count != expected_count:
        raise ValueError(f"{path} row count mismatch: expected {expected_count}, got {total_count}")

    bad_prod = df.filter(F.col("prod_id").isNull() | (~F.col("prod_id").rlike(r"^a_"))).count()
    bad_user = df.filter(F.col("user_id").isNull() | (~F.col("user_id").rlike(r"^u_"))).count()

    if bad_prod > 0:
        raise ValueError(f"{path} has invalid prod_id rows: {bad_prod}")
    if bad_user > 0:
        raise ValueError(f"{path} has invalid user_id rows: {bad_user}")

    bad_votes = df.filter(F.col("votes").isNotNull() & F.col("votes").cast("double").isNull()).count()
    bad_time = df.filter(F.col("time").isNotNull() & F.to_timestamp("time").isNull()).count()

    votes_ok_ratio = 1.0 - (bad_votes / total_count if total_count else 0.0)
    time_ok_ratio = 1.0 - (bad_time / total_count if total_count else 0.0)

    print(f"[L2 {'train' if is_train else 'test'}] votes convertible ratio: {votes_ok_ratio:.4f}")
    print(f"[L2 {'train' if is_train else 'test'}] time convertible ratio:  {time_ok_ratio:.4f}")

    if votes_ok_ratio < 0.95:
        raise ValueError(f"{path} votes convertible ratio too low: {votes_ok_ratio:.4f}")
    if time_ok_ratio < 0.95:
        raise ValueError(f"{path} time convertible ratio too low: {time_ok_ratio:.4f}")

    if is_train:
        valid_rating_count = df.filter(F.col("rating").cast("int").isin([1, 2, 3, 4, 5])).count()
        rating_ok_ratio = valid_rating_count / total_count if total_count else 0.0
        print(f"[L2 train] rating valid ratio: {rating_ok_ratio:.4f}")
        if rating_ok_ratio < 0.95:
            raise ValueError(f"{path} rating valid ratio too low: {rating_ok_ratio:.4f}")

    return total_count


EXPECTED_TRAIN_ROWS = 5_173_005
EXPECTED_TEST_ROWS = None

first_level_test_path = FEATURE_DIR / "final_test.parquet"
if first_level_test_path.exists():
    EXPECTED_TEST_ROWS = spark.read.parquet(str(first_level_test_path)).count()

print("\n=== Layer3 Input Version ===")
print(f"TEXT_VERSION: {TEXT_VERSION}")
print(f"MODEL_TYPE: {MODEL_TYPE}")
print(f"TITLE_VECTOR_COL: {TITLE_VECTOR_COL}")
print(f"COMMENT_VECTOR_COL: {COMMENT_VECTOR_COL}")
print(f"L2_TRAIN_PATH: {L2_TRAIN_PATH}")
print(f"L2_TEST_PATH: {L2_TEST_PATH}")

processor = GoodsFeatures(
    spark=spark,
    train_input_path=L2_TRAIN_PATH,
    test_input_path=L2_TEST_PATH,
    train_output_path=L3_TRAIN_OUT,
    test_output_path=L3_TEST_OUT,
    smoothing_strength=10,
    log_base=2.718,
    force_recompute=FORCE_RECOMPUTE_FEATURES,
)

processor.TRAIN_REQUIRED_COLS = L2_TRAIN_REQUIRED_COLS
processor.TEST_REQUIRED_COLS = L2_TEST_REQUIRED_COLS

if USE_TEST == 1:
    if FORCE_RECOMPUTE_FEATURES or (not L3_TRAIN_CORE_OUT.exists()) or (not L3_VAL_HOLDOUT_OUT.exists()):
        print("🚀 Start strict validation input check...")
        validate_l2_input(L2_TRAIN_PATH, L2_TRAIN_REQUIRED_COLS, is_train=True, expected_count=EXPECTED_TRAIN_ROWS)

        print("🚀 Start strict validation feature engineering...")
        start_time = time.time()

        processor.run_strict_validation(
            train_core_output_path=L3_TRAIN_CORE_OUT,
            val_output_path=L3_VAL_HOLDOUT_OUT,
            holdout_ratio=STRICT_HOLDOUT_RATIO,
        )

        print(f"✅ Strict validation feature engineering finished in {(time.time() - start_time) / 60:.2f} min")
        print(f"📁 train_core saved to: {L3_TRAIN_CORE_OUT}")
        print(f"📁 val_holdout saved to: {L3_VAL_HOLDOUT_OUT}")
    else:
        print("⏭️ Strict validation features already exist, skip recompute.")
        print(f"   use: {L3_TRAIN_CORE_OUT}")
        print(f"   use: {L3_VAL_HOLDOUT_OUT}")
else:
    if FORCE_RECOMPUTE_FEATURES or (not L3_TRAIN_OUT.exists()) or (not L3_TEST_OUT.exists()):
        print("🚀 Start full train/test input check...")
        validate_l2_input(L2_TRAIN_PATH, L2_TRAIN_REQUIRED_COLS, is_train=True, expected_count=EXPECTED_TRAIN_ROWS)
        validate_l2_input(L2_TEST_PATH, L2_TEST_REQUIRED_COLS, is_train=False, expected_count=EXPECTED_TEST_ROWS)

        print("🚀 Start full train/test feature engineering...")
        start_time = time.time()

        processor.run()

        print(f"✅ Full train/test feature engineering finished in {(time.time() - start_time) / 60:.2f} min")
        print(f"📁 train saved to: {L3_TRAIN_OUT}")
        print(f"📁 test saved to: {L3_TEST_OUT}")
    else:
        print("⏭️ Full train/test features already exist, skip recompute.")
        print(f"   use: {L3_TRAIN_OUT}")
        print(f"   use: {L3_TEST_OUT}")
        
        


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import time
import gc
from sklearn.metrics import mean_squared_error, accuracy_score
import pyarrow.dataset as ds

CHUNK_SIZE = 500000
TRAIN_PARQUET = L3_TRAIN_OUT
TEST_PARQUET = L3_TEST_OUT

TITLE_VECTOR_COL = TEXT_VERSION_CONFIG["title_col"]
COMMENT_VECTOR_COL = TEXT_VERSION_CONFIG["comment_col"]

TEMP_DIR = L3_TEMP_ROOT
TEMP_DIR.mkdir(parents=True, exist_ok=True)

MODEL_SAVE_PATH = TEMP_DIR / (
    f"{MODEL_TAG}_debug_{TEXT_VERSION}.{MODEL_FILE_EXT}"
    if USE_TEST == 1
    else f"{MODEL_TAG}_submit_{TEXT_VERSION}.{MODEL_FILE_EXT}"
)
SUBMISSION_PATH = TEMP_DIR / f"submission_{TEXT_VERSION}_{MODEL_TYPE}.csv"

print(f"TEXT_VERSION: {TEXT_VERSION}")
print(f"MODEL_TYPE: {MODEL_TYPE}")
print(f"TITLE_VECTOR_COL: {TITLE_VECTOR_COL}")
print(f"COMMENT_VECTOR_COL: {COMMENT_VECTOR_COL}")
print(f"临时目录: {TEMP_DIR}")
print(f"模型保存路径: {MODEL_SAVE_PATH}")
print(f"提交文件路径: {SUBMISSION_PATH}")

LAYER3_COLS = [
    "votes",
    "purchased",
    "prod_age_days",
    "review_month",
    "day_of_week",
    "u_cnt",
    "u_trust",
    "u_avg_smoothed",
    "u_avg_votes_received",
    "is_super_user",
    "p_cnt",
    "p_avg_smoothed",
    "expectation_gap",
    "abs_expectation_gap",
    "log_votes",
    "log_u_cnt",
    "log_p_cnt",
]

TEXT_STAT_COLS = [
    "title_vec_norm",
    "comment_vec_norm",
    "vec_norm_gap",
    "vec_norm_ratio",
    "title_tfidf_nnz",
    "comment_tfidf_nnz",
    "title_tfidf_l2",
    "comment_tfidf_l2",
    "title_tfidf_max",
    "comment_tfidf_max",
]

REQUIRED_VECTOR_COLS = [TITLE_VECTOR_COL, COMMENT_VECTOR_COL]
REQUIRED_TFIDF_COLS = ["title_tfidf_features", "comment_tfidf_features"]

MIN_VALID_LABEL_RATIO = 0.95
TEXT_SUMMARY_DIM = 1
EXPECTED_TOTAL_DIM = len(LAYER3_COLS) + 300 + TEXT_SUMMARY_DIM + len(TEXT_STAT_COLS)


def extract_to_float32(v, dim=150):
    def fit_dim(vals):
        vals = np.asarray(vals, dtype=np.float32)
        if vals.shape[0] == dim:
            return vals
        arr = np.zeros(dim, dtype=np.float32)
        arr[: min(dim, len(vals))] = vals[:dim]
        return arr

    if hasattr(v, "toArray"):
        return fit_dim(v.toArray())

    if isinstance(v, dict):
        indices = v.get("indices")
        values = v.get("values")

        if indices is not None and values is not None and len(indices) == len(values) and len(indices) > 0:
            arr = np.zeros(dim, dtype=np.float32)
            arr[np.asarray(indices, dtype=np.int32)] = np.asarray(values, dtype=np.float32)
            return arr

        if values is not None:
            return fit_dim(values)

    if isinstance(v, (list, tuple, np.ndarray)):
        return fit_dim(v)

    return np.zeros(dim, dtype=np.float32)


def extract_sparse_stats(v):
    if hasattr(v, "indices") and hasattr(v, "values"):
        vals = np.asarray(v.values, dtype=np.float32)
    elif hasattr(v, "toArray"):
        arr = np.asarray(v.toArray(), dtype=np.float32)
        vals = arr[arr != 0]
    elif isinstance(v, dict):
        values = v.get("values")
        if values is not None:
            vals = np.asarray(values, dtype=np.float32)
        else:
            arr = np.asarray(v.get("data", []), dtype=np.float32)
            vals = arr[arr != 0]
    elif isinstance(v, (list, tuple, np.ndarray)):
        arr = np.asarray(v, dtype=np.float32)
        vals = arr[arr != 0]
    else:
        vals = np.zeros(0, dtype=np.float32)

    if vals.size == 0:
        return np.array([0.0, 0.0, 0.0], dtype=np.float32)

    return np.array(
        [
            float(vals.size),
            float(np.linalg.norm(vals)),
            float(np.max(vals)),
        ],
        dtype=np.float32,
    )


def process_chunk(df, is_train=True):
    raw_size = len(df)
    print(f"   原始块大小: {raw_size}")

    missing_vector_cols = [c for c in REQUIRED_VECTOR_COLS if c not in df.columns]
    if missing_vector_cols:
        raise KeyError(f"缺少向量列: {missing_vector_cols}")

    missing_tfidf_cols = [c for c in REQUIRED_TFIDF_COLS if c not in df.columns]
    if missing_tfidf_cols:
        raise KeyError(f"缺少 TF-IDF 列: {missing_tfidf_cols}")

    if is_train:
        df["rating_clean"] = pd.to_numeric(df["rating"], errors="coerce")
        valid_mask = df["rating_clean"].isin([1, 2, 3, 4, 5])
        valid_count = int(valid_mask.sum())
        valid_ratio = valid_count / raw_size if raw_size else 0.0

        print(f"   rating 有效数: {valid_count}")
        print(f"   rating 有效比例: {valid_ratio:.4f}")

        if valid_ratio < MIN_VALID_LABEL_RATIO:
            raise ValueError(f"当前块有效标签比例过低: {valid_ratio:.4f}")

        df = df.loc[valid_mask].copy()
        y = df["rating_clean"].astype(np.float32).values

    for col in LAYER3_COLS:
        if col not in df.columns:
            raise KeyError(f"缺少数值特征列: {col}")
        df[col] = pd.to_numeric(df[col], errors="coerce")

    null_ratios = df[LAYER3_COLS].isna().mean().round(4).to_dict()
    print(f"   数值列空值比例: {null_ratios}")

    title_vec = np.stack(df[TITLE_VECTOR_COL].apply(extract_to_float32))
    comment_vec = np.stack(df[COMMENT_VECTOR_COL].apply(extract_to_float32))

    title_tfidf_stats = np.stack(df["title_tfidf_features"].apply(extract_sparse_stats))
    comment_tfidf_stats = np.stack(df["comment_tfidf_features"].apply(extract_sparse_stats))

    numeric_feat = df[LAYER3_COLS].fillna(0).values.astype(np.float32)

    title_norm = np.linalg.norm(title_vec, axis=1).astype(np.float32)
    comment_norm = np.linalg.norm(comment_vec, axis=1).astype(np.float32)

    cosine = (
        (title_vec * comment_vec).sum(axis=1) /
        (title_norm * comment_norm + 1e-8)
    ).astype(np.float32)
    cosine = np.where(np.isfinite(cosine), cosine, 0.0).astype(np.float32)[:, None]

    vec_norm_gap = (comment_norm - title_norm).astype(np.float32)[:, None]
    vec_norm_ratio = (comment_norm / (title_norm + 1e-6)).astype(np.float32)[:, None]
    vec_norm_ratio = np.where(np.isfinite(vec_norm_ratio), vec_norm_ratio, 0.0).astype(np.float32)

    text_stats = np.hstack([
        title_norm[:, None],
        comment_norm[:, None],
        vec_norm_gap,
        vec_norm_ratio,
        title_tfidf_stats[:, [0]],
        comment_tfidf_stats[:, [0]],
        title_tfidf_stats[:, [1]],
        comment_tfidf_stats[:, [1]],
        title_tfidf_stats[:, [2]],
        comment_tfidf_stats[:, [2]],
    ]).astype(np.float32)

    X = np.hstack([numeric_feat, title_vec, comment_vec, cosine, text_stats]).astype(np.float32)

    if X.shape[1] != EXPECTED_TOTAL_DIM:
        raise ValueError(f"特征维度异常: 期望 {EXPECTED_TOTAL_DIM}，实际 {X.shape[1]}")

    if is_train:
        return X, y

    return X, df["id"].values


In [ ]:
import time
import gc
import numpy as np
import pandas as pd

print(">>> 开始准备一次性训练数据...")
start_time = time.time()

XGB_DEVICE = "cuda"   # 在 5090 那台机器上跑 XGBoost 时改成 "cuda"

if USE_TEST == 1:
    TRAIN_PARQUET = L3_TRAIN_CORE_OUT
    VAL_PARQUET = L3_VAL_HOLDOUT_OUT
    TEST_PARQUET = None

    train_x_path = TEMP_DIR / f"train_core_X_{TEXT_VERSION}_{MODEL_TYPE}.float32.memmap"
    train_y_path = TEMP_DIR / f"train_core_y_{TEXT_VERSION}_{MODEL_TYPE}.float32.memmap"
    val_x_path = TEMP_DIR / f"val_holdout_X_{TEXT_VERSION}_{MODEL_TYPE}.float32.memmap"
    val_y_path = TEMP_DIR / f"val_holdout_y_{TEXT_VERSION}_{MODEL_TYPE}.float32.memmap"
else:
    TRAIN_PARQUET = L3_TRAIN_OUT
    VAL_PARQUET = None
    TEST_PARQUET = L3_TEST_OUT

    train_x_path = TEMP_DIR / f"train_full_X_{TEXT_VERSION}_{MODEL_TYPE}.float32.memmap"
    train_y_path = TEMP_DIR / f"train_full_y_{TEXT_VERSION}_{MODEL_TYPE}.float32.memmap"
    val_x_path = None
    val_y_path = None


def build_supervised_memmap(parquet_path, x_path, y_path, tag):
    dataset = ds.dataset(str(parquet_path), format="parquet")
    row_capacity = int(dataset.count_rows())

    if row_capacity <= 0:
        raise ValueError(f"{tag} has no rows: {parquet_path}")

    for p in [x_path, y_path]:
        if p.exists():
            p.unlink()

    X_mm = np.memmap(
        x_path,
        mode="w+",
        dtype=np.float32,
        shape=(row_capacity, EXPECTED_TOTAL_DIM),
    )
    y_mm = np.memmap(
        y_path,
        mode="w+",
        dtype=np.float32,
        shape=(row_capacity,),
    )

    filled = 0

    for batch_idx, batch in enumerate(dataset.to_batches(batch_size=CHUNK_SIZE), 1):
        print(f"\n--- {tag} 第 {batch_idx} 块数据 ---")
        chunk_df = batch.to_pandas()
        X_chunk, y_chunk = process_chunk(chunk_df, is_train=True)

        del chunk_df, batch
        gc.collect()

        row_count = len(X_chunk)
        if row_count == 0:
            print(f"   [跳过] {tag} 当前块为空。")
            continue

        X_mm[filled: filled + row_count] = X_chunk
        y_mm[filled: filled + row_count] = y_chunk
        filled += row_count

        print(f"   {tag} 已累计样本: {filled}")

        del X_chunk, y_chunk
        gc.collect()

    if filled == 0:
        raise ValueError(f"{tag} has no valid rows after processing")

    X_mm.flush()
    y_mm.flush()

    return X_mm, y_mm, filled


def build_model(model_type):
    if model_type == "x":
        return xgb.XGBRegressor(
            objective="reg:squarederror",
            eval_metric="rmse",
            n_estimators=200,
            learning_rate=0.05,
            max_depth=8,
            min_child_weight=20,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=2.0,
            reg_alpha=0.0,
            tree_method="hist",
            device=XGB_DEVICE,
            random_state=42,
            n_jobs=-1,
        )

    if model_type == "l":
        try:
            import lightgbm as lgb
        except ImportError as e:
            raise ImportError(
                "MODEL_TYPE='l' 需要先安装 lightgbm。当前环境未安装。"
            ) from e

        return lgb.LGBMRegressor(
            objective="regression",
            n_estimators=1000,
            learning_rate=0.03,
            num_leaves=63,
            max_depth=-1,
            min_child_samples=50,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=2.0,
            reg_alpha=0.0,
            random_state=42,
            n_jobs=-1,
            force_col_wise=True,
            verbosity=-1,
        )

    raise ValueError(f"unknown MODEL_TYPE: {model_type}")


def save_trained_model(model, model_type, save_path):
    if model_type == "x":
        model.save_model(str(save_path))
    elif model_type == "l":
        model.booster_.save_model(str(save_path))
    else:
        raise ValueError(f"unknown MODEL_TYPE: {model_type}")


print(">>> 构建训练矩阵...")
X_train_mm, y_train_mm, train_rows = build_supervised_memmap(
    TRAIN_PARQUET,
    train_x_path,
    train_y_path,
    tag="训练集",
)

X_train = X_train_mm[:train_rows]
y_train = y_train_mm[:train_rows]

print(f"train_rows: {train_rows}")
train_mean = float(np.mean(y_train))
print(f"train_mean: {train_mean:.4f}")

if USE_TEST == 1:
    print("\n>>> 构建严格验证矩阵...")
    X_val_mm, y_val_mm, val_rows = build_supervised_memmap(
        VAL_PARQUET,
        val_x_path,
        val_y_path,
        tag="验证集",
    )
    X_val = X_val_mm[:val_rows]
    y_val = y_val_mm[:val_rows]
    print(f"val_rows: {val_rows}")
else:
    X_val = None
    y_val = None
    val_rows = 0

print(f"\n训练矩阵准备完成，耗时: {(time.time() - start_time) / 60:.2f} 分钟")

# 1. 初始化模型
model = build_model(MODEL_TYPE)
# 获取你在 Cell 1 中定义的迭代次数
n_epochs = N_ESTIMATORS if 'N_ESTIMATORS' in locals() else 100

# --- 2. 训练阶段 (Training) 计时 ---
print(f"\n>>> 开始模型训练: {MODEL_TAG} (迭代次数: {n_epochs})")
train_start = time.time()

model.fit(X_train, y_train)

train_end = time.time()
total_train_time_sec = train_end - train_start

# --- 3. 推理阶段 (Inference) 计时 ---
if USE_TEST == 0:
    print("\n>>> 开始 Kaggle 测试集分块推理...")
    test_dataset = ds.dataset(str(TEST_PARQUET), format="parquet")
    all_preds, all_ids = [], []
    
    infer_start = time.time()
    
    # 保持原分块推理逻辑不变
    for batch in test_dataset.to_batches(batch_size=CHUNK_SIZE):
        chunk_df = batch.to_pandas()
        X_test_chunk, test_ids = process_chunk(chunk_df, is_train=False)
        
        preds = model.predict(X_test_chunk)
        all_preds.extend(np.clip(preds, 1.0, 5.0).tolist())
        all_ids.extend(test_ids)
        
        del chunk_df, X_test_chunk, batch
        gc.collect()

    infer_end = time.time()
    total_infer_time_sec = infer_end - infer_start
    
    # 保存提交文件 (原功能)
    submission = pd.DataFrame({"id": all_ids, "rating": all_preds})
    submission.to_csv(SUBMISSION_PATH, index=False)
    print(f"预测完成，提交文件已保存: {SUBMISSION_PATH}")

# --- 4. 最终效率报告打印 (直接用于 Project Report) ---
# 自动引用上一个单元格记录的预处理时间
if 'total_prep_time_sec' not in locals():
    total_prep_time_sec = 0

print("\n" + "="*45)
print("   COMP5434 PROJECT EFFICIENCY EVALUATION")
print("="*45)
# 指标 1: 每轮训练时间 (Training time per epoch)
print(f"1. Training time per epoch: {total_train_time_sec / n_epochs:.4f} seconds")

# 指标 2: 总离线时间 (Preprocessing + Training)
total_offline_time = total_prep_time_sec + total_train_time_sec
print(f"2. Total offline time: {total_offline_time:.2f} seconds ({(total_offline_time/60):.2f} mins)")
print(f"   - Breakdown: Preprocessing={total_prep_time_sec:.2f}s, Training={total_train_time_sec:.2f}s")

# 指标 3: 总推理时间 (Total inference time)
if 'total_infer_time_sec' in locals():
    print(f"3. Total inference time: {total_infer_time_sec:.2f} seconds")
else:
    print("3. Total inference time: N/A (仅在预测模式下统计)")
print("="*45)


In [ ]:
print(f">>> {TEXT_VERSION} / {TRAIN_PARQUET.name} 抽样预览")

preview_columns = [
    "id", "rating", "votes", "review_month", "day_of_week",
    "u_trust", "p_avg_smoothed", "expectation_gap",
]

preview_dataset = ds.dataset(str(TRAIN_PARQUET), format="parquet")
preview_batch = next(
    preview_dataset.to_batches(
        batch_size=5,
        columns=preview_columns,
    )
)

print(preview_batch.to_pandas())
